In [1]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

PyTorch Version: 2.14.1+cu126
CUDA Available: True
GPU Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# ==============================================================================
# NEXUS OptiML 2026 — GPU-Accelerated & Temporal-Aligned Pipeline
# Models: XGBoost (GPU), CatBoost (GPU), AdaBoost, Random Forest (GPU/XGBRF)
# Designed for Local CV == Kaggle LB Alignment & 0.59+ ROC-AUC Generalization
# ==============================================================================

import os
import re
import gc
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, PowerTransformer, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

import xgboost as xgb
from catboost import CatBoostClassifier, Pool
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
np.random.seed(SEED)

# Detect GPU availability for XGBoost, CatBoost, and cuML Random Forest
try:
    import torch
    USE_GPU = torch.cuda.is_available()
except ImportError:
    USE_GPU = os.environ.get("CUDA_VISIBLE_DEVICES", "") != ""

print(f"GPU Acceleration Enabled: {USE_GPU}")

# ==============================================================================
# 1. DATA LOADING
# ==============================================================================
DATA_DIR = "."  # Set to "/kaggle/input/nexus-opti-ml-2026-operation-insight" on Kaggle

def load_all_tables(data_dir=DATA_DIR):
    stores = pd.read_csv(os.path.join(data_dir, "stores.csv"))
    calendar = pd.read_csv(os.path.join(data_dir, "calendar.csv"))
    customers = pd.read_csv(os.path.join(data_dir, "customers.csv"))
    products = pd.read_csv(os.path.join(data_dir, "products.csv"))
    test = pd.read_csv(os.path.join(data_dir, "test.csv"))
    sub = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))
    
    train_path = os.path.join(data_dir, "train.csv")
    if os.path.exists(train_path):
        train = pd.read_csv(train_path)
        print(f"Loaded train.csv: {train.shape} | test.csv: {test.shape}")
    else:
        raise FileNotFoundError(
            f"train.csv not found in {data_dir}. Please attach the competition dataset."
        )
    return train, test, stores, calendar, customers, products, sub

train_df, test_df, stores_df, calendar_df, customers_df, products_df, sub_df = load_all_tables()

# ==============================================================================
# 2. CANONICAL TEXT NORMALIZATION & SUPPLEMENTARY DATASET PROCESSING
# ==============================================================================
def clean_region_name(val):
    if pd.isna(val):
        return "Unknown"
    s = re.sub(r"[^a-z0-9]", "", str(val).strip().lower())
    mapping = {
        "brazilsoutheast": "Brazil-Southeast", "brazilsutheast": "Brazil-Southeast",
        "brazilsouth": "Brazil-South",
        "brazilnortheast": "Brazil-Northeast", "brazilnrtheast": "Brazil-Northeast",
        "brazilcentralwest": "Brazil-CentralWest", "brazilcetralwest": "Brazil-CentralWest",
        "brazilnorth": "Brazil-North",
        "uknorth": "UK-North", "uknrth": "UK-North",
        "uksouth": "UK-South", "uksuth": "UK-South",
        "ukmidlands": "UK-Midlands", "ukmilands": "UK-Midlands",
        "uswest": "US-West",
        "useast": "US-East", "usest": "US-East", "usast": "US-East",
        "uscentral": "US-Central", "uscetral": "US-Central",
        "ussouth": "US-South", "ussuth": "US-South",
        "westerneurope": "Western Europe",
        "southerneurope": "Southern Europe", "southereurope": "Southern Europe",
        "nordics": "Nordics", "norics": "Nordics",
        "asiapacific": "Asia Pacific", "asiapcific": "Asia Pacific",
        "ireland": "Ireland", "ireand": "Ireland",
        "otherinternational": "Other International", "otherintrnational": "Other International",
        "americas": "Americas", "amercas": "Americas",
        "myanmaryangon": "Myanmar-Yangon",
        "myanmarmandalay": "Myanmar-Mandalay", "myanmarandalay": "Myanmar-Mandalay",
        "myanmarnaypyitaw": "Myanmar-Naypyitaw", "myanmaraypyitaw": "Myanmar-Naypyitaw",
    }
    return mapping.get(s, str(val).strip())

def clean_category_name(val):
    if pd.isna(val):
        return "unknown"
    s = re.sub(r"[^a-z0-9]", "", str(val).strip().lower())
    mapping = {
        "homedecor": "home_decor", "homeecor": "home_decor",
        "kitchendining": "kitchen_dining",
        "sportsleisure": "sports_leisure", "sportseisure": "sports_leisure",
        "toysgifts": "toys_gifts", "toysifts": "toys_gifts",
        "fashionaccessories": "fashion_accessories", "fashionacessories": "fashion_accessories",
        "officesupplies": "office_supplies", "officeupplies": "office_supplies",
        "healthbeauty": "health_beauty",
        "technology": "technology", "technlogy": "technology",
        "electronicsaccessories": "electronics_accessories",
        "gardendiy": "garden_diy", "gardediy": "garden_diy",
        "furniture": "furniture", "furnture": "furniture",
        "foodbeverages": "food_beverages", "foodbeerages": "food_beverages",
    }
    return mapping.get(s, str(val).strip().lower())

def prepare_reference_tables(stores, customers, products, calendar):
    # --- 1. STORES ---
    stores = stores.copy()
    stores["region"] = stores["region"].apply(clean_region_name)
    # 100% of online & marketplace stores have NaN size_sqm (virtual stores)
    stores["is_virtual_store"] = stores["size_sqm"].isna().astype(np.int8)
    stores["size_sqm"] = stores.groupby(["channel", "format"])["size_sqm"].transform(
        lambda x: x.fillna(x.median())
    ).fillna(0.0)
    stores["opened_date"] = pd.to_datetime(stores["opened_date"])
    # Stationary store vintage rank (0 to 1) instead of drifting raw day count
    stores["store_vintage_pct"] = stores["opened_date"].rank(pct=True).astype(np.float32)
    # Drop stores['channel'] when merging since it is 100% identical to orders['channel']
    stores.drop(columns=["channel", "opened_date"], inplace=True)
    
    # --- 2. CUSTOMERS ---
    customers = customers.copy()
    customers["region_cust"] = customers["region"].apply(clean_region_name)
    customers.drop(columns=["region"], inplace=True)
    customers["signup_date"] = pd.to_datetime(customers["signup_date"])
    customers["record_updated_at"] = pd.to_datetime(customers["record_updated_at"])
    
    tier_map = {"bronze": 1, "silver": 2, "gold": 3, "platinum": 4}
    customers["loyalty_rank"] = customers["loyalty_tier"].map(tier_map).fillna(1).astype(np.int8)
    
    # Sort chronologically by customer_id and record_updated_at
    customers = customers.sort_values(["customer_id", "record_updated_at"])
    customers["age_band"] = customers.groupby("customer_id")["age_band"].ffill().bfill()
    
    # Capture multi-record customer loyalty changes across snapshots
    cust_dyn = customers.groupby("customer_id").agg(
        cust_snapshot_count=("record_updated_at", "count"),
        first_loyalty_rank=("loyalty_rank", "first"),
        last_loyalty_rank=("loyalty_rank", "last"),
    ).reset_index()
    cust_dyn["loyalty_tier_delta"] = (
        cust_dyn["last_loyalty_rank"] - cust_dyn["first_loyalty_rank"]
    ).astype(np.int8)
    
    customers = customers.drop_duplicates(subset=["customer_id"], keep="last").copy()
    customers = customers.merge(
        cust_dyn[["customer_id", "cust_snapshot_count", "loyalty_tier_delta"]],
        on="customer_id", how="left"
    )
    
    # Relational imputation of missing age_band via (customer_type, loyalty_tier, region_cust) mode
    customers["age_band_was_null"] = customers["age_band"].isna().astype(np.int8)
    for grp in [["customer_type", "loyalty_tier", "region_cust"], ["customer_type", "loyalty_tier"]]:
        mode_s = (
            customers.dropna(subset=["age_band"])
            .groupby(grp)["age_band"]
            .agg(lambda x: x.value_counts().index[0])
        )
        customers = customers.set_index(grp)
        customers["age_band"] = customers["age_band"].fillna(mode_s)
        customers = customers.reset_index()
    customers["age_band"] = customers["age_band"].fillna("25-34")
    
    age_num_map = {"18-24": 21.0, "25-34": 30.0, "35-44": 40.0, "45-54": 50.0, "55-64": 60.0, "65+": 70.0}
    customers["age_midpoint"] = customers["age_band"].map(age_num_map).fillna(35.0).astype(np.float32)
    
    # --- 3. PRODUCTS ---
    products = products.copy()
    products["category"] = products["category"].apply(clean_category_name)
    products["launch_date"] = pd.to_datetime(products["launch_date"])
    products["prod_launch_pct"] = products["launch_date"].rank(pct=True).astype(np.float32)
    
    products["volume_cm3"] = (
        products["length_cm"].clip(lower=1.0)
        * products["width_cm"].clip(lower=1.0)
        * products["height_cm"].clip(lower=1.0)
    ).astype(np.float32)
    products["max_dim_cm"] = products[["length_cm", "width_cm", "height_cm"]].max(axis=1).astype(np.float32)
    products["min_dim_cm"] = products[["length_cm", "width_cm", "height_cm"]].min(axis=1).astype(np.float32)
    products["aspect_ratio"] = (products["max_dim_cm"] / products["min_dim_cm"].clip(lower=1.0)).astype(np.float32)
    
    # Physical density imputation for weight_kg using (category, packaging_type)
    products["weight_was_null"] = (products["weight_kg"].isna() | (products["weight_kg"] <= 0)).astype(np.int8)
    valid_w = products["weight_kg"] > 0
    products["density"] = np.where(valid_w, (products["weight_kg"] * 1000.0) / products["volume_cm3"], np.nan)
    med_dens = products.groupby(["category", "packaging_type"])["density"].transform("median")
    med_dens = med_dens.fillna(products["density"].median())
    
    products["weight_kg"] = np.where(
        valid_w, products["weight_kg"], (products["volume_cm3"] * med_dens) / 1000.0
    ).astype(np.float32)
    products["density"] = ((products["weight_kg"] * 1000.0) / products["volume_cm3"]).astype(np.float32)
    products["list_price_per_kg"] = (products["list_price"] / products["weight_kg"].clip(lower=0.05)).astype(np.float32)
    
    # Category-relative product benchmarks
    cat_ref = products.groupby("category").agg(
        cat_med_price=("list_price", "median"),
        cat_med_weight=("weight_kg", "median"),
        cat_med_vol=("volume_cm3", "median"),
    ).reset_index()
    products = products.merge(cat_ref, on="category", how="left")
    products["prod_price_vs_cat"] = (products["list_price"] / products["cat_med_price"].clip(lower=0.1)).astype(np.float32)
    products["prod_weight_vs_cat"] = (products["weight_kg"] / products["cat_med_weight"].clip(lower=0.01)).astype(np.float32)
    products["prod_vol_vs_cat"] = (products["volume_cm3"] / products["cat_med_vol"].clip(lower=1.0)).astype(np.float32)
    products.drop(columns=["launch_date", "cat_med_price", "cat_med_weight", "cat_med_vol"], inplace=True)
    
    # --- 4. CALENDAR (STATIONARY TRANSFORMATION TO FIX 2023-2024 vs 2025 DRIFT) ---
    calendar = calendar.copy()
    calendar["week_start"] = pd.to_datetime(calendar["week_start"])
    calendar = calendar.sort_values("week_start").reset_index(drop=True)
    
    # Replace drifting absolute CPI and Unemployment levels with stationary 1w/4w changes & rolling ratios
    calendar["cpi_pct_change_1w"] = calendar["cpi"].pct_change(1).fillna(0.0).astype(np.float32)
    calendar["cpi_pct_change_4w"] = calendar["cpi"].pct_change(4).fillna(0.0).astype(np.float32)
    calendar["unemp_diff_4w"] = calendar["unemployment"].diff(4).fillna(0.0).astype(np.float32)
    calendar["fuel_vs_4w_ma"] = (
        calendar["fuel_price"] / calendar["fuel_price"].rolling(4, min_periods=1).mean()
    ).astype(np.float32)
    calendar["surcharge_vs_4w_ma"] = (
        calendar["fuel_surcharge"] / calendar["fuel_surcharge"].rolling(4, min_periods=1).mean()
    ).astype(np.float32)
    calendar["temp_diff_1w"] = calendar["temperature_f"].diff(1).fillna(0.0).astype(np.float32)
    # Drop raw non-stationary cpi and unemployment so trees cannot overfit to 2023-2024 ranges
    calendar.drop(columns=["cpi", "unemployment", "fuel_price"], inplace=True)
    
    return stores, customers, products, calendar

stores_ref, customers_ref, products_ref, calendar_ref = prepare_reference_tables(
    stores_df, customers_df, products_df, calendar_df
)

# ==============================================================================
# 3. RELATIONAL NULL IMPUTATION, OUTLIER REPAIR & STATIONARY FEATURE ENGINEERING
# ==============================================================================
def engineer_features(train, test, stores, customers, products, calendar):
    print("Performing relational imputation, outlier repair, and stationary feature engineering...")
    n_train = len(train)
    train["is_test"] = 0
    test["is_test"] = 1
    test["returned"] = np.nan
    
    df = pd.concat([train, test], axis=0, ignore_index=True)
    
    # --- A. EXACT DETERMINISTIC & RELATIONAL NULL IMPUTATION ---
    # 1. Propagate order-invariant attributes within order_id
    for col in ["tz_hint", "payment_method", "ship_mode", "session_clicks"]:
        df[f"{col}_was_null"] = df[col].isna().astype(np.int8)
        df[col] = df.groupby("order_id")[col].transform(lambda x: x.ffill().bfill())
        
    # 2. tz_hint is 99.8% deterministic by store_id!
    store_tz_map = (
        df.dropna(subset=["tz_hint"])
        .groupby("store_id")["tz_hint"]
        .agg(lambda x: x.value_counts().index[0])
    )
    df["tz_hint"] = df["tz_hint"].fillna(df["store_id"].map(store_tz_map)).fillna("UTC")
    
    # 3. ship_mode: channel == 'store' is 100% 'in_store'; then customer_id preference; then (channel, return_window_days)
    df.loc[df["channel"] == "store", "ship_mode"] = df.loc[df["channel"] == "store", "ship_mode"].fillna("in_store")
    cust_sm_map = (
        df.dropna(subset=["ship_mode"])
        .groupby("customer_id")["ship_mode"]
        .agg(lambda x: x.value_counts().index[0])
    )
    df["ship_mode"] = df["ship_mode"].fillna(df["customer_id"].map(cust_sm_map))
    ch_sm_map = (
        df.dropna(subset=["ship_mode"])
        .groupby(["channel", "return_window_days"])["ship_mode"]
        .agg(lambda x: x.value_counts().index[0])
    )
    df = df.set_index(["channel", "return_window_days"])
    df["ship_mode"] = df["ship_mode"].fillna(ch_sm_map)
    df = df.reset_index()
    df["ship_mode"] = df["ship_mode"].fillna("standard")
    
    # 4. payment_method: customer_id historical preference -> (channel, device_type) mode
    cust_pm_map = (
        df.dropna(subset=["payment_method"])
        .groupby("customer_id")["payment_method"]
        .agg(lambda x: x.value_counts().index[0])
    )
    df["payment_method"] = df["payment_method"].fillna(df["customer_id"].map(cust_pm_map))
    ch_pm_map = (
        df.dropna(subset=["payment_method"])
        .groupby(["channel", "device_type"])["payment_method"]
        .agg(lambda x: x.value_counts().index[0])
    )
    df = df.set_index(["channel", "device_type"])
    df["payment_method"] = df["payment_method"].fillna(ch_pm_map)
    df = df.reset_index()
    df["payment_method"] = df["payment_method"].fillna("card")
    
    # 5. Coupon indicators (drop raw coupon_code string to prevent birthday-paradox overfitting)
    df["has_coupon"] = df["coupon_code"].notna().astype(np.int8)
    df["order_has_coupon"] = df.groupby("order_id")["has_coupon"].transform("max").astype(np.int8)
    df.drop(columns=["coupon_code"], inplace=True)
    
    # --- B. MERGE SUPPLEMENTARY TABLES ---
    df = df.merge(products, on="product_id", how="left")
    df = df.merge(stores, on="store_id", how="left")
    df = df.merge(customers, on="customer_id", how="left")
    
    df["order_dt"] = pd.to_datetime(df["order_ts"])
    df["week_start"] = df["order_dt"].dt.to_period("W-SUN").dt.start_time
    min_wk, max_wk = calendar["week_start"].min(), calendar["week_start"].max()
    df["week_start_clipped"] = df["week_start"].clip(lower=min_wk, upper=max_wk)
    df = df.merge(calendar, left_on="week_start_clipped", right_on="week_start", how="left", suffixes=("", "_cal"))
    
    # --- C. FIX 100x UNIT_PRICE CENTS ANOMALY & EXACT DISCOUNT IMPUTATION ---
    raw_line_spend = df["unit_price"] * df["quantity"].clip(lower=1)
    cents_bug = (
        (df["unit_price"] / df["list_price"].clip(lower=0.01) > 8.0)
        | ((df["order_value"] > 0) & ((df["order_value"] / raw_line_spend) < 0.05))
    )
    df["is_cents_bug"] = cents_bug.astype(np.int8)
    df["clean_unit_price"] = np.where(cents_bug, df["unit_price"] / 100.0, df["unit_price"]).astype(np.float32)
    
    # Impute missing discount via exact product list_price formula (r = 0.9737)
    df["discount_was_null"] = df["discount"].isna().astype(np.int8)
    implied_disc = np.clip(
        1.0 - (df["clean_unit_price"] / (0.963 * df["list_price"].clip(lower=0.01))), 0.0, 0.80
    ).astype(np.float32)
    df["discount"] = df["discount"].fillna(implied_disc).astype(np.float32)
    df["discount_residual"] = (df["discount"] - implied_disc).astype(np.float32)
    
    # Impute session_clicks via (device_type, channel, items_in_order decile)
    sc_med = df.groupby(["device_type", "channel"])["session_clicks"].transform("median")
    df["session_clicks"] = df["session_clicks"].fillna(sc_med).fillna(9.0).astype(np.float32)
    
    # Impute freight_cost via physical line weight + ship_mode + channel
    df["freight_was_null"] = df["freight_cost"].isna().astype(np.int8)
    df["line_weight_kg"] = (df["weight_kg"] * df["quantity"]).astype(np.float32)
    df["line_volume_cm3"] = (df["volume_cm3"] * df["quantity"]).astype(np.float32)
    w_bin = pd.qcut(df["line_weight_kg"].rank(method="first"), q=10, labels=False)
    fr_med = df.groupby(["ship_mode", "channel", w_bin])["freight_cost"].transform("median")
    df["freight_cost"] = df["freight_cost"].fillna(fr_med).fillna(df["freight_cost"].median()).astype(np.float32)
    
    # --- D. FINANCIAL, LOGISTICS & ORDER-STRUCTURE FEATURES ---
    df["expected_line_value"] = (df["clean_unit_price"] * df["quantity"]).astype(np.float32)
    df["order_val_discrepancy_ratio"] = (
        df["order_value"] / df["expected_line_value"].clip(lower=0.01)
    ).astype(np.float32)
    df["unit_to_list_ratio"] = (df["clean_unit_price"] / df["list_price"].clip(lower=0.01)).astype(np.float32)
    df["freight_to_val_ratio"] = (df["freight_cost"] / (df["order_value"] + 1.0)).astype(np.float32)
    df["freight_per_kg"] = (df["freight_cost"] / (df["line_weight_kg"] + 0.1)).astype(np.float32)
    df["surcharge_adj_freight"] = (df["freight_cost"] / (1.0 + df["fuel_surcharge"].fillna(0.3))).astype(np.float32)
    df["dollar_discount_total"] = (
        (df["list_price"] - df["clean_unit_price"]).clip(lower=0.0) * df["quantity"]
    ).astype(np.float32)
    
    # Intra-Order Aggregations
    ord_stats = df.groupby("order_id").agg(
        obs_order_lines=("order_line_id", "count"),
        ord_sum_val=("order_value", "sum"),
        ord_mean_val=("order_value", "mean"),
        ord_std_val=("order_value", "std"),
        ord_max_val=("order_value", "max"),
        ord_sum_qty=("quantity", "sum"),
        ord_sum_weight=("line_weight_kg", "sum"),
        ord_sum_freight=("freight_cost", "sum"),
        ord_mean_disc=("discount", "mean"),
        ord_max_disc=("discount", "max"),
        ord_n_cats=("category", "nunique"),
        ord_n_stores=("store_id", "nunique"),
        ord_n_promos=("promo_channel_code", "nunique"),
        ord_min_dt=("order_dt", "min"),
        ord_max_dt=("order_dt", "max"),
    ).reset_index()
    ord_stats["ord_std_val"] = ord_stats["ord_std_val"].fillna(0.0).astype(np.float32)
    ord_stats["ord_ts_spread_mins"] = (
        (ord_stats["ord_max_dt"] - ord_stats["ord_min_dt"]).dt.total_seconds() / 60.0
    ).astype(np.float32)
    ord_stats["is_multi_store_order"] = (ord_stats["ord_n_stores"] > 1).astype(np.int8)
    ord_stats.drop(columns=["ord_min_dt", "ord_max_dt"], inplace=True)
    df = df.merge(ord_stats, on="order_id", how="left")
    
    df["line_val_share_of_order"] = (df["order_value"] / df["ord_sum_val"].clip(lower=0.01)).astype(np.float32)
    df["line_val_vs_ord_max"] = (df["order_value"] / df["ord_max_val"].clip(lower=0.01)).astype(np.float32)
    df["line_weight_share_of_order"] = (df["line_weight_kg"] / df["ord_sum_weight"].clip(lower=0.01)).astype(np.float32)
    df["line_no_rel_pos"] = (df["line_no"] / df["items_in_order"].clip(lower=1)).astype(np.float32)
    df["obs_to_reported_lines_ratio"] = (df["obs_order_lines"] / df["items_in_order"].clip(lower=1)).astype(np.float32)
    df["clicks_per_line"] = (df["session_clicks"] / df["obs_order_lines"].clip(lower=1)).astype(np.float32)
    
    # Geographic & Cross-Table Alignment
    df["same_region_cust_store"] = (df["region_cust"] == df["region"]).astype(np.int8)
    df["cust_country"] = df["region_cust"].str.split("-").str[0]
    df["store_country"] = df["region"].str.split("-").str[0]
    df["same_country_cust_store"] = (df["cust_country"] == df["store_country"]).astype(np.int8)
    
    # Stationary Temporal & Lifecycle Features
    df["order_hour"] = df["order_dt"].dt.hour.astype(np.int8)
    df["order_dow"] = df["order_dt"].dt.dayofweek.astype(np.int8)
    df["is_weekend"] = (df["order_dow"] >= 5).astype(np.int8)
    df["hour_sin"] = np.sin(2.0 * np.pi * df["order_hour"] / 24.0).astype(np.float32)
    df["hour_cos"] = np.cos(2.0 * np.pi * df["order_hour"] / 24.0).astype(np.float32)
    df["dow_sin"] = np.sin(2.0 * np.pi * df["order_dow"] / 7.0).astype(np.float32)
    df["dow_cos"] = np.cos(2.0 * np.pi * df["order_dow"] / 7.0).astype(np.float32)
    
    # Log-transformed customer tenure & snapshot recency relative to order_dt
    raw_tenure = (df["order_dt"] - df["signup_date"]).dt.days.clip(lower=0).fillna(180)
    df["log_cust_tenure_days"] = np.log1p(raw_tenure).astype(np.float32)
    df["is_new_customer_30d"] = (raw_tenure <= 30).astype(np.int8)
    df["updated_after_order"] = (df["record_updated_at"] > df["order_dt"]).astype(np.int8)
    
    # --- E. STATIONARY PER-ENTITY BEHAVIORAL PROFILES (NORMALIZED PER PERIOD) ---
    # Normalize order counts to per-month rates so 24-month train & 6-month test have identical scales!
    period_months = np.where(df["is_test"] == 0, 24.0, 6.0)
    df["period_months"] = period_months
    
    cust_prof = df.groupby("customer_id").agg(
        cust_mean_val=("order_value", "mean"),
        cust_max_val=("order_value", "max"),
        cust_mean_disc=("discount", "mean"),
        cust_mean_qty=("quantity", "mean"),
        cust_gift_rate=("gift_wrap", "mean"),
        cust_mean_ret_win=("return_window_days", "mean"),
        cust_mean_clicks=("session_clicks", "mean"),
        cust_n_cats=("category", "nunique"),
    ).reset_index()
    df = df.merge(cust_prof, on="customer_id", how="left")
    df["val_vs_cust_mean"] = (df["order_value"] / df["cust_mean_val"].clip(lower=0.1)).astype(np.float32)
    df["disc_vs_cust_mean"] = (df["discount"] - df["cust_mean_disc"]).astype(np.float32)
    
    prod_prof = df.groupby("product_id").agg(
        prod_mean_disc=("discount", "mean"),
        prod_mean_qty=("quantity", "mean"),
        prod_gift_rate=("gift_wrap", "mean"),
        prod_mean_unit_price=("clean_unit_price", "mean"),
    ).reset_index()
    df = df.merge(prod_prof, on="product_id", how="left")
    df["unit_price_vs_prod_mean"] = (
        df["clean_unit_price"] / df["prod_mean_unit_price"].clip(lower=0.01)
    ).astype(np.float32)
    df["disc_vs_prod_mean"] = (df["discount"] - df["prod_mean_disc"]).astype(np.float32)
    
    store_prof = df.groupby("store_id").agg(
        store_mean_val=("order_value", "mean"),
        store_mean_disc=("discount", "mean"),
        store_mean_freight=("freight_cost", "mean"),
        store_gift_rate=("gift_wrap", "mean"),
    ).reset_index()
    df = df.merge(store_prof, on="store_id", how="left")
    
    # Keep order_dt timestamp on train for Temporal Validation diagnostics before dropping
    train_ts = df.loc[df["is_test"] == 0, "order_dt"].reset_index(drop=True)
    
    drop_cols = [
        "unit_price", "order_ts", "order_dt", "week_start", "week_start_clipped",
        "week_start_cal", "signup_date", "record_updated_at", "period_months"
    ]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    
    train_out = df[df["is_test"] == 0].drop(columns=["is_test"]).reset_index(drop=True)
    test_out = df[df["is_test"] == 1].drop(columns=["is_test", "returned"]).reset_index(drop=True)
    
    return train_out, test_out, train_ts

train_fe, test_fe, train_order_dt = engineer_features(
    train_df, test_df, stores_ref, customers_ref, products_ref, calendar_ref
)
del train_df, test_df
gc.collect()

# ==============================================================================
# 4. LEAK-FREE ENCODING, WINSORIZATION & FEATURE MATRICES
# ==============================================================================
# Notice: High-cardinality IDs (customer_id, product_id) are NOT passed as raw tree features!
ID_COLS = ["order_line_id", "order_id", "customer_id", "product_id"]
TARGET_COL = "returned"

CAT_COLS = [
    "store_id", "supplier_code", "category", "packaging_type", "color",
    "tz_hint", "channel", "payment_method", "ship_mode", "promo_channel_code",
    "warehouse_zone", "picker_shift", "device_type", "region", "format",
    "region_cust", "customer_type", "age_band", "preferred_contact",
    "loyalty_tier", "cust_country", "store_country", "banner_id"
]
CAT_COLS = [c for c in CAT_COLS if c in train_fe.columns]
NUM_COLS = [c for c in train_fe.columns if c not in ID_COLS + [TARGET_COL] + CAT_COLS]

def encode_and_clean_outliers(train, test, cat_cols, num_cols, n_splits=5):
    print("Applying Customer-Grouped OOF Target Encoding & Outlier Winsorization...")
    y = train[TARGET_COL].astype(int).values
    global_prior = y.mean()
    
    # GroupKFold by customer_id ensures zero customer or order leakage across folds!
    gkf = GroupKFold(n_splits=n_splits)
    cust_groups = train["customer_id"].values
    
    # Only target-encode shared structural entities with strong smoothing (m = 100)
    te_cols = [
        "store_id", "supplier_code", "category", "banner_id", "warehouse_zone",
        "region_cust", "region", "ship_mode", "payment_method", "channel",
        "device_type", "format", "promo_channel_code"
    ]
    te_cols = [c for c in te_cols if c in cat_cols]
    smoothing = 100.0
    
    for col in te_cols:
        oof_te = np.full(len(train), global_prior, dtype=np.float32)
        for tr_idx, val_idx in gkf.split(train, y, groups=cust_groups):
            stats = train.iloc[tr_idx].groupby(col)[TARGET_COL].agg(["count", "mean"])
            smooth = (stats["count"] * stats["mean"] + smoothing * global_prior) / (stats["count"] + smoothing)
            oof_te[val_idx] = train.iloc[val_idx][col].map(smooth).fillna(global_prior).values
        train[f"{col}_te"] = oof_te
        
        full_stats = train.groupby(col)[TARGET_COL].agg(["count", "mean"])
        full_smooth = (full_stats["count"] * full_stats["mean"] + smoothing * global_prior) / (full_stats["count"] + smoothing)
        test[f"{col}_te"] = test[col].map(full_smooth).fillna(global_prior).astype(np.float32)
        
    all_num_cols = num_cols + [f"{c}_te" for c in te_cols]
    
    # Winsorize extreme numerical outliers at [0.5%, 99.5%] based strictly on train percentiles
    for col in all_num_cols:
        train[col] = train[col].replace([np.inf, -np.inf], np.nan)
        test[col] = test[col].replace([np.inf, -np.inf], np.nan)
        med = float(train[col].median())
        train[col] = train[col].fillna(med).astype(np.float32)
        test[col] = test[col].fillna(med).astype(np.float32)
        
        if train[col].nunique() > 15:
            q_low, q_high = np.percentile(train[col], [0.5, 99.5])
            if q_high > q_low:
                train[col] = np.clip(train[col], q_low, q_high).astype(np.float32)
                test[col] = np.clip(test[col], q_low, q_high).astype(np.float32)
                
    # Ordinal encode categoricals for XGBoost / AdaBoost / Random Forest
    ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.int32)
    train[cat_cols] = ord_enc.fit_transform(train[cat_cols].astype(str))
    test[cat_cols] = ord_enc.transform(test[cat_cols].astype(str))
    
    return train, test, all_num_cols

train_fe, test_fe, ALL_NUM_COLS = encode_and_clean_outliers(
    train_fe, test_fe, CAT_COLS, NUM_COLS, n_splits=5
)

FEATURES = ALL_NUM_COLS + CAT_COLS
y_all = train_fe[TARGET_COL].astype(int).values
cust_groups_all = train_fe["customer_id"].values

print(f"Final Feature Count: {len(FEATURES)} ({len(ALL_NUM_COLS)} numeric, {len(CAT_COLS)} categorical)")

# ==============================================================================
# 5. TEMPORAL VALIDATION DIAGNOSTIC (VERIFYING LOCAL CV == KAGGLE LB)
# ==============================================================================
# Sort by order_dt and hold out the final 20% of training time (Months 21-24)
# to verify out-of-time generalization before full 5-fold training.
time_cutoff = train_order_dt.quantile(0.80)
tr_time_mask = (train_order_dt <= time_cutoff).values
va_time_mask = (train_order_dt > time_cutoff).values
print(f"Temporal Split Cutoff Date: {time_cutoff} | Train: {tr_time_mask.sum()} | Future Val: {va_time_mask.sum()}")

# ==============================================================================
# 6. OPTUNA HYPERPARAMETER TUNING ON TEMPORAL SPLIT (XGBOOST GPU)
# ==============================================================================
def tune_xgb_on_temporal_split(X, y, tr_mask, va_mask, n_trials=12):
    print(f"\n--- Tuning XGBoost (GPU) on Out-of-Time Temporal Split ({n_trials} trials) ---")
    X_tr, y_tr = X.iloc[tr_mask], y[tr_mask]
    X_va, y_va = X.iloc[va_mask], y[va_mask]
    
    def objective(trial):
        params = {
            "n_estimators": 1500,
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.06, log=True),
            "max_depth": trial.suggest_int("max_depth", 5, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 40, 200),
            "subsample": trial.suggest_float("subsample", 0.65, 0.90),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.85),
            "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 20.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 2.0, 30.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.05, 2.0, log=True),
            "eval_metric": "auc",
            "tree_method": "hist",
            "device": "cuda" if USE_GPU else "cpu",
            "early_stopping_rounds": 50,
            "random_state": SEED,
        }
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        preds = model.predict_proba(X_va)[:, 1]
        return roc_auc_score(y_va, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, timeout=600)
    print(f"Out-of-Time Temporal Validation ROC-AUC (Matches Kaggle LB): {study.best_value:.5f}")
    print("Best Tuned XGBoost Params:", study.best_params)
    return study.best_params

best_xgb_params = tune_xgb_on_temporal_split(
    train_fe[FEATURES], y_all, tr_time_mask, va_time_mask, n_trials=12
)

# ==============================================================================
# 7. TRAIN THE 4 MODELS: XGBOOST (GPU), CATBOOST (GPU), ADABOOST, RANDOM FOREST (GPU)
# ==============================================================================
def train_four_models(X, y, X_test, groups, best_xgb_params, n_splits=5):
    oof_xgb = np.zeros(len(y), dtype=np.float32)
    pred_xgb = np.zeros(len(X_test), dtype=np.float32)
    
    oof_cat = np.zeros(len(y), dtype=np.float32)
    pred_cat = np.zeros(len(X_test), dtype=np.float32)
    
    oof_rf = np.zeros(len(y), dtype=np.float32)
    pred_rf = np.zeros(len(X_test), dtype=np.float32)
    
    oof_ada = np.zeros(len(y), dtype=np.float32)
    pred_ada = np.zeros(len(X_test), dtype=np.float32)
    
    # Time-decay sample weights: give recent transactions (Months 13-24) up to 1.5x higher weight
    # so models prioritize recent customer/product return patterns closest to Months 25-30!
    time_rank = train_order_dt.rank(pct=True).values
    sample_weights = (0.75 + 0.75 * time_rank).astype(np.float32)
    
    gkf = GroupKFold(n_splits=n_splits)
    
    xgb_cfg = {
        **best_xgb_params,
        "n_estimators": 2500,
        "eval_metric": "auc",
        "tree_method": "hist",
        "device": "cuda" if USE_GPU else "cpu",
        "early_stopping_rounds": 60,
    }
    
    cat_cfg = {
        "iterations": 2500,
        "learning_rate": 0.045,
        "depth": 7,
        "l2_leaf_reg": 10.0,
        "eval_metric": "AUC",
        "task_type": "GPU" if USE_GPU else "CPU",
        "random_seed": SEED,
        "early_stopping_rounds": 60,
        "verbose": 0,
    }
    
    # GPU-Accelerated Random Forest via XGBRFClassifier (native CUDA histogram RF)
    rf_cfg = {
        "n_estimators": 400,
        "max_depth": 10,
        "min_child_weight": 60,
        "subsample": 0.80,
        "colsample_bynode": 0.65,
        "reg_lambda": 5.0,
        "eval_metric": "auc",
        "tree_method": "hist",
        "device": "cuda" if USE_GPU else "cpu",
        "n_jobs": -1,
    }
    
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
        print(f"\n================ FOLD {fold + 1} / {n_splits} ================")
        X_tr, y_tr, w_tr = X.iloc[tr_idx], y[tr_idx], sample_weights[tr_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]
        
        # 1. XGBoost (GPU)
        m_xgb = xgb.XGBClassifier(**xgb_cfg, random_state=SEED + fold)
        m_xgb.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof_xgb[val_idx] = m_xgb.predict_proba(X_va)[:, 1]
        pred_xgb += m_xgb.predict_proba(X_test)[:, 1] / n_splits
        print(f"Fold {fold+1} XGBoost (GPU)  ROC-AUC: {roc_auc_score(y_va, oof_xgb[val_idx]):.5f}")
        
        # 2. CatBoost (GPU)
        m_cat = CatBoostClassifier(**cat_cfg)
        m_cat.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=(X_va, y_va), verbose=False)
        oof_cat[val_idx] = m_cat.predict_proba(X_va)[:, 1]
        pred_cat += m_cat.predict_proba(X_test)[:, 1] / n_splits
        print(f"Fold {fold+1} CatBoost (GPU) ROC-AUC: {roc_auc_score(y_va, oof_cat[val_idx]):.5f}")
        
        # 3. Random Forest (GPU-accelerated via XGBRFClassifier)
        m_rf = xgb.XGBRFClassifier(**rf_cfg, random_state=SEED + fold)
        m_rf.fit(X_tr, y_tr, sample_weight=w_tr, verbose=False)
        oof_rf[val_idx] = m_rf.predict_proba(X_va)[:, 1]
        pred_rf += m_rf.predict_proba(X_test)[:, 1] / n_splits
        print(f"Fold {fold+1} Random Forest  ROC-AUC: {roc_auc_score(y_va, oof_rf[val_idx]):.5f}")
        
        # 4. AdaBoost (Trained on stratified 250k subsample per fold for fast execution on 1.4M rows)
        ada_idx = np.random.RandomState(SEED + fold).choice(
            len(tr_idx), size=min(250_000, len(tr_idx)), replace=False
        )
        m_ada = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, random_state=SEED + fold),
            n_estimators=80,
            learning_rate=0.08,
            random_state=SEED + fold,
        )
        m_ada.fit(X_tr.iloc[ada_idx], y_tr[ada_idx], sample_weight=w_tr[ada_idx])
        oof_ada[val_idx] = m_ada.predict_proba(X_va)[:, 1]
        pred_ada += m_ada.predict_proba(X_test)[:, 1] / n_splits
        print(f"Fold {fold+1} AdaBoost       ROC-AUC: {roc_auc_score(y_va, oof_ada[val_idx]):.5f}")
        
        gc.collect()
        
    print("\n================ FULL OOF VALIDATION SUMMARY ================")
    print(f"Full OOF XGBoost (GPU)  ROC-AUC: {roc_auc_score(y, oof_xgb):.5f}")
    print(f"Full OOF CatBoost (GPU) ROC-AUC: {roc_auc_score(y, oof_cat):.5f}")
    print(f"Full OOF Random Forest  ROC-AUC: {roc_auc_score(y, oof_rf):.5f}")
    print(f"Full OOF AdaBoost       ROC-AUC: {roc_auc_score(y, oof_ada):.5f}")
    
    return (oof_xgb, pred_xgb), (oof_cat, pred_cat), (oof_rf, pred_rf), (oof_ada, pred_ada)

(oof_xgb, pred_xgb), (oof_cat, pred_cat), (oof_rf, pred_rf), (oof_ada, pred_ada) = train_four_models(
    train_fe[FEATURES], y_all, test_fe[FEATURES], cust_groups_all, best_xgb_params, n_splits=5
)

# ==============================================================================
# 8. CLEAN DIRECT PROBABILITY BLEND (NO META-STACKER OVERFITTING)
# ==============================================================================
# Fixed, transparent weights favoring the strongest GPU gradient boosters while
# incorporating variance reduction from Random Forest and AdaBoost:
W_XGB = 0.45
W_CAT = 0.40
W_RF  = 0.10
W_ADA = 0.05

oof_blend = (
    W_XGB * oof_xgb
    + W_CAT * oof_cat
    + W_RF  * oof_rf
    + W_ADA * oof_ada
)
test_blend = (
    W_XGB * pred_xgb
    + W_CAT * pred_cat
    + W_RF  * pred_rf
    + W_ADA * pred_ada
)

print(f"\n>>> Final Clean Blend OOF ROC-AUC (All 24 Months): {roc_auc_score(y_all, oof_blend):.5f} <<<")
print(f">>> Final Clean Blend Future-Window ROC-AUC (Months 21-24): {roc_auc_score(y_all[va_time_mask], oof_blend[va_time_mask]):.5f} <<<")

# ==============================================================================
# 9. SUBMISSION GENERATION
# ==============================================================================
submission = pd.DataFrame({
    "order_line_id": test_fe["order_line_id"].values,
    "returned": np.clip(test_blend, 0.0, 1.0)
})

submission = sub_df[["order_line_id"]].merge(submission, on="order_line_id", how="left")
assert len(submission) == 479995, f"Expected 479,995 rows, got {len(submission)}"
assert submission["returned"].isna().sum() == 0, "Submission contains NaNs!"

submission.to_csv("submission10.csv", index=False)
print("\nSuccessfully wrote 'submission10.csv'! Summary:")
print(submission["returned"].describe())

c:\python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU Acceleration Enabled: True
Loaded train.csv: (1400005, 27) | test.csv: (479995, 26)
Performing relational imputation, outlier repair, and stationary feature engineering...
Applying Customer-Grouped OOF Target Encoding & Outlier Winsorization...
Final Feature Count: 155 (132 numeric, 23 categorical)
Temporal Split Cutoff Date: 2024-08-07 09:44:00 | Train: 1120017 | Future Val: 279988

--- Tuning XGBoost (GPU) on Out-of-Time Temporal Split (12 trials) ---
Out-of-Time Temporal Validation ROC-AUC (Matches Kaggle LB): 0.67414
Best Tuned XGBoost Params: {'learning_rate': 0.054662776851447166, 'max_depth': 6, 'min_child_weight': 170, 'subsample': 0.6728285330845745, 'colsample_bytree': 0.7390912789055076, 'reg_alpha': 13.70659788029049, 'reg_lambda': 8.78398805956775, 'gamma': 0.054473330242750105}

================ FOLD 1 / 5 ================
Fold 1 XGBoost (GPU)  ROC-AUC: 0.73503


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1 CatBoost (GPU) ROC-AUC: 0.73697
Fold 1 Random Forest  ROC-AUC: 0.72033
Fold 1 AdaBoost       ROC-AUC: 0.71835

================ FOLD 2 / 5 ================
Fold 2 XGBoost (GPU)  ROC-AUC: 0.73619


Default metric period is 5 because AUC is/are not implemented for GPU


...

Fold 3 CatBoost (GPU) ROC-AUC: 0.73653
Fold 3 Random Forest  ROC-AUC: 0.71223
Fold 3 AdaBoost       ROC-AUC: 0.71689

================ FOLD 4 / 5 ================
Fold 4 XGBoost (GPU)  ROC-AUC: 0.72747


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4 CatBoost (GPU) ROC-AUC: 0.73150
Fold 4 Random Forest  ROC-AUC: 0.71339
Fold 4 AdaBoost       ROC-AUC: 0.71259

================ FOLD 5 / 5 ================
Fold 5 XGBoost (GPU)  ROC-AUC: 0.73209


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5 CatBoost (GPU) ROC-AUC: 0.73879
Fold 5 Random Forest  ROC-AUC: 0.71778
Fold 5 AdaBoost       ROC-AUC: 0.71600

================ FULL OOF VALIDATION SUMMARY ================
Full OOF XGBoost (GPU)  ROC-AUC: 0.73012
Full OOF CatBoost (GPU) ROC-AUC: 0.73671
Full OOF Random Forest  ROC-AUC: 0.71537
Full OOF AdaBoost       ROC-AUC: 0.71626

>>> Final Clean Blend OOF ROC-AUC (All 24 Months): 0.73426 <<<
>>> Final Clean Blend Future-Window ROC-AUC (Months 21-24): 0.70416 <<<

Successfully wrote 'submission.csv'! Summary:
count    479995.000000
mean          0.178467
std           0.102290
min           0.025885
25%           0.106210
50%           0.151720
75%           0.222092
max           0.864471
Name: returned, dtype: float64
